In [ ]:
# ===== NB11 | CELL 1: header =====
from google.colab import drive
drive.mount('/content/drive')
print('CELL 1 DONE')

In [ ]:
# ===== NB11 | CELL 2: write the evaluation helper (evallib.py) =====
# WHAT: save functions for bootstrap confidence intervals, ensemble averaging and subgroup AUCs.
# WHY:  with one seed, small score differences can be luck. Confidence intervals show which differences are real.
#       Notebooks 13 to 17 reuse this file.
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
CODE = r'''"""Evaluation helpers: bootstrap confidence intervals, ensemble averaging, subgroup AUCs."""
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


def avg_probs(proj, runs, split):
    """Average the saved probabilities of several runs (the ensemble)."""
    from toxlib import load_probs
    return np.mean([load_probs(proj, r, split) for r in runs], axis=0)


def ci95(a):
    lo, hi = np.percentile(a, [2.5, 97.5])
    return float(lo), float(hi)


def paired_bootstrap_macro_ap(Y, probs, n_boot=300, seed=42):
    """probs = {run: P}. The same resampled rows are used for every run, so differences are meaningful."""
    rng = np.random.RandomState(seed)
    out = {k: [] for k in probs}
    for _ in range(n_boot):
        idx = rng.randint(0, len(Y), len(Y))
        for k, P in probs.items():
            out[k].append(np.mean([average_precision_score(Y[idx, i], P[idx, i]) for i in range(len(LABELS))]))
    return {k: np.array(v) for k, v in out.items()}


def bootstrap_metrics(Y, P, thr, n_boot=500, seed=42):
    """Per-label precision, recall, F1, AP with 95% bootstrap intervals at fixed thresholds, plus macro F1 / macro AP."""
    t = np.array([thr[l] for l in LABELS])

    def metrics(y, p):
        pred = p >= t
        tp = (pred & (y == 1)).sum(0); fp = (pred & (y == 0)).sum(0); fn = ((~pred) & (y == 1)).sum(0)
        prec = tp / np.maximum(tp + fp, 1); rec = tp / np.maximum(tp + fn, 1)
        f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
        ap = np.array([average_precision_score(y[:, i], p[:, i]) for i in range(len(LABELS))])
        return np.vstack([prec, rec, f1, ap])

    base = metrics(Y, P)
    rng = np.random.RandomState(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.randint(0, len(Y), len(Y))
        boots.append(metrics(Y[idx], P[idx]))
    boots = np.array(boots)
    names = ['precision', 'recall', 'F1', 'AP']
    rows = []
    for i, l in enumerate(LABELS):
        r = {'label': l, 'n_pos': int(Y[:, i].sum()), 'threshold': t[i]}
        for j, m in enumerate(names):
            lo, hi = np.percentile(boots[:, j, i], [2.5, 97.5])
            r[m] = round(float(base[j, i]), 3)
            r[m + '_95CI'] = f'{lo:.3f}-{hi:.3f}'
        rows.append(r)
    macro = {'macro_F1': round(float(base[2].mean()), 4), 'macro_F1_95CI': [round(x, 4) for x in ci95(boots[:, 2, :].mean(1))],
             'macro_AP': round(float(base[3].mean()), 4), 'macro_AP_95CI': [round(x, 4) for x in ci95(boots[:, 3, :].mean(1))]}
    return pd.DataFrame(rows), macro


def auc_safe(y, s):
    y = np.asarray(y)
    if len(y) < 2 or y.min() == y.max():
        return np.nan
    return roc_auc_score(y, s)


def subgroup_aucs(y, s, mask):
    """Subgroup AUC, BPSN AUC and BNSP AUC for one identity mask (y = 0/1 labels, s = scores)."""
    sub, bg = mask, ~mask
    sub_auc = auc_safe(y[sub], s[sub])
    sel = (sub & (y == 0)) | (bg & (y == 1)); bpsn = auc_safe(y[sel], s[sel])
    sel = (sub & (y == 1)) | (bg & (y == 0)); bnsp = auc_safe(y[sel], s[sel])
    return sub_auc, bpsn, bnsp


def bpsn_ci(y, s, mask, n_boot=200, seed=42):
    """95% interval for BPSN AUC, resampling the two groups separately (fast)."""
    rng = np.random.RandomState(seed)
    neg = s[mask & (y == 0)]; pos = s[(~mask) & (y == 1)]
    if len(neg) < 5 or len(pos) < 5:
        return (np.nan, np.nan)
    vals = []
    for _ in range(n_boot):
        a = rng.choice(neg, len(neg)); b = rng.choice(pos, len(pos))
        vals.append(roc_auc_score(np.r_[np.zeros(len(a)), np.ones(len(b))], np.r_[a, b]))
    return ci95(vals)
'''
import sys, importlib
open(f'{PROJECT_DIR}/src/evallib.py', 'w').write(CODE)
sys.path.insert(0, f'{PROJECT_DIR}/src')
import evallib; importlib.reload(evallib)
print('evallib written and imported.')
print('CELL 2 DONE')

In [ ]:
# ===== NB11 | CELL 3: loss comparison on validation + decision =====
# WHAT: compare A1, A2, B1_256, B1_128 (plain BCE), C1_128 (weighted) and C2_128 (focal) on validation,
#       with bootstrap intervals, training curves and per-label AP. Then pick the loss for RoBERTa.
# WHY:  the brief asks for class weighting OR focal loss. We test both against plain BCE and choose with evidence.
#       Rule: the winner is the better of C1 and C2 (the brief requires one of them). Plain BCE is the reference.
import sys, importlib, json, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, evallib
for m in (toxlib, evallib): importlib.reload(m)
from toxlib import LABELS, load_split, load_probs, macro_ap, tune_thresholds, per_label_report
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

for d in ['graphs', 'tables', 'final', 'fairness', 'explainability']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)
Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
WANT = ['A1', 'A2', 'B1_256', 'B1_128', 'C1_128', 'C2_128']
RUNS = [r for r in WANT if os.path.exists(f'{PROJECT_DIR}/results/predictions/{r}_val.npy')]
print('Runs found:', RUNS, '| missing:', [r for r in WANT if r not in RUNS])
P = {r: load_probs(PROJECT_DIR, r, 'val') for r in RUNS}
neural = [r for r in RUNS if r not in ('A1', 'A2')]
boot = evallib.paired_bootstrap_macro_ap(Yva, {r: P[r] for r in neural}, n_boot=300)
ref = 'B1_128' if 'B1_128' in neural else neural[0]

rows, per_ap = [], {}
for r in RUNS:
    thr = tune_thresholds(Yva, P[r]); rep = per_label_report(Yva, P[r], thr)
    per_ap[r] = rep.set_index('label').AP
    row = {'run': r, 'Macro AP': round(macro_ap(Yva, P[r]), 4), 'Macro F1 tuned': round(rep.F1.mean(), 4)}
    if r in boot:
        lo, hi = evallib.ci95(boot[r]); row['AP 95% CI'] = f'{lo:.3f}-{hi:.3f}'
        if r != ref:
            d = boot[r] - boot[ref]; dlo, dhi = evallib.ci95(d)
            row[f'diff vs {ref}'] = f'{d.mean():+.4f} ({dlo:+.4f} to {dhi:+.4f})'
    rows.append(row)
T = pd.DataFrame(rows)
print('\nT09 loss comparison (validation):'); print(T.to_string(index=False))
print('A difference is only real if its interval does NOT include 0.')
A = pd.DataFrame(per_ap).round(4); print('\nAverage Precision per label:'); print(A)
T.to_csv(f'{PROJECT_DIR}/results/tables/T09_loss_comparison.csv', index=False)
A.to_csv(f'{PROJECT_DIR}/results/tables/T09_per_label_AP.csv')

# F7 training curves
plt.figure(figsize=(7, 4))
for r in ['B1_128', 'C1_128', 'C2_128']:
    p = f'{PROJECT_DIR}/results/experiment_logs/{r}.json'
    if os.path.exists(p):
        h = json.load(open(p))['history']
        plt.plot([e['epoch'] for e in h], [e['val_macro_ap'] for e in h], marker='o', label=r)
p = f'{PROJECT_DIR}/results/experiment_logs/B1_distilbert_bce.json'
if os.path.exists(p):
    try:
        h = json.load(open(p))['epochs']
        plt.plot([e['epoch'] for e in h], [e['macro_ap'] for e in h], marker='x', ls='--', color='gray', label='B1_256 (notebook 04)')
    except Exception as e:
        print('(skipped the B1_256 curve:', e, ')')
plt.xlabel('epoch'); plt.ylabel('validation Macro AP'); plt.title('F7: training curves'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{PROJECT_DIR}/results/graphs/F7_training_curves.png', dpi=150); plt.show()
# F8 per-label AP
A.plot(kind='bar', figsize=(10, 4)); plt.ylabel('validation AP'); plt.title('F8: per-label AP by run'); plt.xticks(rotation=25)
plt.tight_layout(); plt.savefig(f'{PROJECT_DIR}/results/graphs/F8_per_label_AP.png', dpi=150); plt.show()

# decision: the winner is chosen between C1 and C2
cand = [r for r in ['C1_128', 'C2_128'] if r in P]
assert cand, 'C1_128 / C2_128 predictions not found - run notebooks 09 and 10 first'
winner = max(cand, key=lambda r: macro_ap(Yva, P[r]))
decision = {'winner_run': winner, 'winner_loss': {'C1_128': 'weighted', 'C2_128': 'focal'}[winner],
            'val_macro_ap': round(macro_ap(Yva, P[winner]), 4),
            'plain_bce_val_macro_ap': round(macro_ap(Yva, P['B1_128']), 4) if 'B1_128' in P else None}
json.dump(decision, open(f'{PROJECT_DIR}/results/experiment_logs/loss_decision.json', 'w'), indent=2)
print('\nDecision saved:', decision)
print('CELL 3 DONE')